# Data Census
## How much data supports each part of the analysis, and where is it missing?

The objective is to count observations across season, school-calendar regime, monitoring zone and pollutant, and examine whether missingness or imputation could distort the comparisons.

**Data access.** The SIMA measurements used here are not
redistributed in this repository (SIMA provided them for this study on the condition
that they are not disseminated); the repository holds the code, the official school
calendar and all derived results. This notebook reads files that notebooks 00 and 01
build from those measurements, and its configuration cell stops with a clear message
if they are missing. To rebuild them, request the annual workbooks from SIMA, place
them in `data_raw/` and run notebook 00 (its header gives the exact file format).

**Role in the analysis.** Diagnostic: data census and balance checks (coverage, imputed cells, days per season and regime). The analytical hierarchy of the whole repository is stated in the README.

## 1. Initial setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

contaminantes = ['CO', 'NO2', 'O3', 'PM10', 'PM2.5']

output_dir = Path('../data_processed')
censo_dir = output_dir / 'censo_datos'
censo_dir.mkdir(exist_ok=True)

print('Libraries imported successfully')

# Check required inputs (restricted data; see heading)
# Input-data check (restricted data; see heading)
archivos_requeridos = [output_dir / 'base_horaria_estaciones_fisicas.csv.gz',
                       output_dir / 'base_horaria_zonas.csv.gz',
                       output_dir / 'datos_horarios_imputados_hibrido.csv.gz',
                       output_dir / 'panel_pico_hibrido.csv',
                       output_dir / 'calendario_oficial_sep.csv']
faltantes = [str(a) for a in archivos_requeridos if not a.exists()]
if faltantes:
    raise FileNotFoundError(
        ''
        ''
        "Files derived from the SIMA measurements are missing; they are not redistributed in this repository. "
        "Request the annual workbooks from SIMA, place them in data_raw/ and run notebooks 00 and 01. "
        f"Missing: {faltantes}")
print(f"Input data found: {len(archivos_requeridos)} files")

Libraries imported successfully
Input data found: 5 files


## 2. Loading data from the pipeline built in notebook 00

In [2]:
# Observed hourly series by zone and physical station, including gaps
base = pd.read_csv('../data_processed/base_horaria_zonas.csv.gz', parse_dates=['date'])
base_fisicas = pd.read_csv('../data_processed/base_horaria_estaciones_fisicas.csv.gz', parse_dates=['date'])

# Hybrid-imputed hourly series, retaining the imputation indicator
imputado = pd.read_csv('../data_processed/datos_horarios_imputados_hibrido.csv.gz', parse_dates=['date'])

# Final analysis panel and official calendar (notebooks 00 and 01)
panel = pd.read_csv('../data_processed/panel_pico_hibrido.csv', parse_dates=['date'])
calendario = pd.read_csv('../data_processed/calendario_oficial_sep.csv', parse_dates=['date'])

print("=" * 70)
print('DATA LOADED')
print("=" * 70)
print(f"Hourly zone base (observed): {base.shape[0]:,} rows")
print(f"Hourly physical-station base: {base_fisicas.shape[0]:,} rows")
print(f"Imputed hourly series (hybrid): {imputado.shape[0]:,} rows")
print(f"Final analysis panel: {panel.shape[0]:,} day-zone rows")

DATA LOADED
Hourly zone base (observed): 772,426 rows
Hourly physical-station base: 1,751,750 rows
Imputed hourly series (hybrid): 789,120 rows
Final analysis panel: 6,576 day-zone rows


## 3. Sensitivity of zone consolidation: mean versus median across stations

Each hourly zone value averages its physical stations. How much would the analysis cell change if the median were used instead? This is a descriptive check on aggregation sensitivity; it does not by itself quantify changes in regression estimates.

In [3]:
base_fisicas['fecha'] = base_fisicas['date'].dt.normalize()
celda_media = (base_fisicas.groupby(['zona', 'fecha', 'parametro'])['valor'].mean())
celda_mediana = (base_fisicas.groupby(['zona', 'fecha', 'parametro'])['valor'].median())
comparacion = pd.concat([celda_media.rename('media'), celda_mediana.rename('mediana')], axis=1).dropna()

print("=" * 70)
print('DAY-ZONE AGGREGATION SENSITIVITY (mean vs median across stations)')
print("=" * 70)
filas_robustez = []
for parametro, sub in comparacion.groupby(level=2):
    correlacion = sub['media'].corr(sub['mediana'])
    dif_relativa = (abs(sub['media'] - sub['mediana']) / sub['mediana'].replace(0, np.nan)).median() * 100
    filas_robustez.append({'parametro': parametro, 'corr_media_mediana': round(correlacion, 4),
                           'dif_relativa_mediana_pct': round(dif_relativa, 1)})
    print(f"  {parametro}: corr = {correlacion:.4f} | median relative difference = {dif_relativa:.1f}%")
corrs = [f['corr_media_mediana'] for f in filas_robustez]
print(f"\n=> Correlations of {min(corrs):.2f} to {max(corrs):.2f}: both aggregation methods show similar temporal profiles.")
print('   The within-day gap cancels differences between mean and median aggregation only if')
print('   those level differences affect both windows proportionally;')
print('   cancellation is not guaranteed by the design.')
pd.DataFrame(filas_robustez).to_csv(censo_dir / 'robustez_consolidacion_media_vs_mediana.csv', index=False)
print(f"Saved to: {censo_dir / 'robustez_consolidacion_media_vs_mediana.csv'}")

DAY-ZONE AGGREGATION SENSITIVITY (mean vs median across stations)
  CO: corr = 0.9621 | median relative difference = 5.5%
  NO2: corr = 0.9747 | median relative difference = 8.7%
  O3: corr = 0.8804 | median relative difference = 12.9%
  PM10: corr = 0.9678 | median relative difference = 7.1%
  PM2.5: corr = 0.9416 | median relative difference = 8.7%

=> Correlations of 0.88 to 0.97: both aggregation methods show similar temporal profiles.
   The within-day gap cancels differences between mean and median aggregation only if
   those level differences affect both windows proportionally;
   cancellation is not guaranteed by the design.
Saved to: ../data_processed/censo_datos/robustez_consolidacion_media_vs_mediana.csv


## 4. Observed hourly coverage by zone and pollutant

Coverage is measured against the full expected grid of hours, zones and pollutants. Missing hours are the gaps targeted by imputation. The filling step retains observed hours, but the pollutant-specific flooring step in notebook 00 can also affect observed values below the floor.

In [4]:
fecha_min, fecha_max = base['date'].min(), base['date'].max()
malla_horas = pd.date_range(fecha_min, fecha_max, freq='h')
horas_esperadas = len(malla_horas)

validos = base.dropna(subset=['valor']).groupby(['estacion', 'parametro'])['valor'].count()
tabla_cobertura = validos.unstack('parametro').reindex(columns=contaminantes)
tabla_faltante_pct = ((1 - tabla_cobertura / horas_esperadas) * 100).round(1)

print("=" * 70)
print(f"OBSERVED COVERAGE ({fecha_min.date()} to {fecha_max.date()}: {horas_esperadas:,} expected hours per cell)")
print("=" * 70)
print('\nValid hours by zone and pollutant:')
print(tabla_cobertura)
print('\n% missing hours (gaps filled by imputation):')
print(tabla_faltante_pct)

tabla_cobertura.to_csv(censo_dir / 'horas_validas_por_estacion_contaminante.csv')
tabla_faltante_pct.to_csv(censo_dir / 'pct_faltante_por_estacion_contaminante.csv')
print(f"\nSaved to: {censo_dir / 'horas_validas_por_estacion_contaminante.csv'}")
print(f"Saved to: {censo_dir / 'pct_faltante_por_estacion_contaminante.csv'}")

OBSERVED COVERAGE (2023-01-01 to 2025-12-31: 26,304 expected hours per cell)

Valid hours by zone and pollutant:
parametro     CO    NO2     O3   PM10  PM2.5
estacion                                    
CENTRO     24184  25498  25270  25690  22365
NORESTE3   25999  25526  26245  26272  25538
NOROESTE3  26254  26037  26112  26267  22563
NORTE2     26252  26257  26112  26233  26017
SUR        26275  26276  26274  26276  26275
SUROESTE2  26219  26271  26150  26243  25476

% missing hours (gaps filled by imputation):
parametro   CO  NO2   O3  PM10  PM2.5
estacion                             
CENTRO     8.1  3.1  3.9   2.3   15.0
NORESTE3   1.2  3.0  0.2   0.1    2.9
NOROESTE3  0.2  1.0  0.7   0.1   14.2
NORTE2     0.2  0.2  0.7   0.3    1.1
SUR        0.1  0.1  0.1   0.1    0.1
SUROESTE2  0.3  0.1  0.6   0.2    3.1

Saved to: ../data_processed/censo_datos/horas_validas_por_estacion_contaminante.csv
Saved to: ../data_processed/censo_datos/pct_faltante_por_estacion_contaminante.csv


In [5]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(tabla_faltante_pct, annot=True, fmt='.1f', cmap='Reds', vmin=0,
            cbar_kws={'label': '% missing hours'}, linewidths=0.5, linecolor='white', ax=ax)
ax.set_xlabel('Pollutant', fontsize=12, fontweight='bold')
ax.set_ylabel('Monitoring zone', fontsize=12, fontweight='bold')
ax.set_title('% missing hours in observed data\n(before imputation)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(censo_dir / '01_pct_faltante_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {censo_dir / '01_pct_faltante_heatmap.png'}")

Plot saved to: ../data_processed/censo_datos/01_pct_faltante_heatmap.png


## 5. Does missingness concentrate in particular months? Zone-by-month coverage

In [6]:
base_mes = base.copy()
base_mes['anio_mes'] = base_mes['date'].dt.to_period('M').astype(str)
malla_mes = pd.Series(malla_horas).dt.to_period('M').astype(str).value_counts().sort_index()

validos_mes = (base_mes.dropna(subset=['valor'])
               .groupby(['estacion', 'anio_mes'])['valor'].count()
               .unstack('anio_mes'))
esperadas_mes = malla_mes * len(contaminantes)
pct_faltante_mes = (1 - validos_mes.div(esperadas_mes, axis=1)) * 100

fig, ax = plt.subplots(figsize=(16, 5.5))
sns.heatmap(pct_faltante_mes, cmap='Reds', vmin=0, vmax=100,
            cbar_kws={'label': '% missing (five pollutants pooled)'}, ax=ax)
ax.set_xlabel('Month', fontsize=12, fontweight='bold')
ax.set_ylabel('Monitoring zone', fontsize=12, fontweight='bold')
ax.set_title('% missing hours by zone and month', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(censo_dir / '02_pct_faltante_por_mes_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

pct_faltante_mes.round(1).to_csv(censo_dir / 'pct_faltante_por_estacion_mes.csv')
print(f"Saved to: {censo_dir / 'pct_faltante_por_estacion_mes.csv'}")
print(f"Plot saved to: {censo_dir / '02_pct_faltante_por_mes_heatmap.png'}")

Saved to: ../data_processed/censo_datos/pct_faltante_por_estacion_mes.csv
Plot saved to: ../data_processed/censo_datos/02_pct_faltante_por_mes_heatmap.png


## 6. Arrival-window census: valid hours per cell and percentage imputed

Each arrival-window day–zone–pollutant cell uses two hourly records, labelled 07 and 08. Against the complete expected grid, we count cells with two, one or zero valid observed hours. Zero observed hours means the entire cell was supplied by imputation.

In [7]:
base['fecha'] = base['date'].dt.normalize()
base['hora'] = base['date'].dt.hour
pico_obs = base[base['hora'].isin([7, 8])].dropna(subset=['valor'])

horas_unicas = (pico_obs.groupby(['fecha', 'estacion', 'parametro'])['hora'].nunique()
                .rename('horas_validas'))

fechas_todas = pd.date_range(base['fecha'].min(), base['fecha'].max(), freq='D')
zonas_todas = sorted(base['estacion'].unique())
malla_completa = pd.MultiIndex.from_product([fechas_todas, zonas_todas, contaminantes],
                                            names=['fecha', 'estacion', 'parametro'])
censo_celdas = horas_unicas.reindex(malla_completa).fillna(0).astype(int).rename('horas_validas').to_frame()

distribucion = censo_celdas['horas_validas'].value_counts().sort_index()
print("=" * 70)
print('VALID HOURS PER DAY-ZONE-POLLUTANT CELL (arrival window; 2 possible)')
print("=" * 70)
for horas, n in distribucion.items():
    print(f"  {horas} valid hours: {n:,} cells ({n/len(censo_celdas)*100:.1f}%)")

VALID HOURS PER DAY-ZONE-POLLUTANT CELL (arrival window; 2 possible)
  0 valid hours: 595 cells (1.8%)
  1 valid hours: 273 cells (0.8%)
  2 valid hours: 32,012 cells (97.4%)


In [8]:
# Does fully imputed-cell frequency vary across regimes and seasons?
censo_celdas = censo_celdas.reset_index()
mapa_clase = calendario.set_index('date')['lectivo']
censo_celdas['clase'] = censo_celdas['fecha'].map(mapa_clase)
mapa_temporada4 = {12: 'invierno', 1: 'invierno', 2: 'invierno',
                   3: 'primavera', 4: 'primavera', 5: 'primavera',
                   6: 'verano', 7: 'verano', 8: 'verano',
                   9: 'otono', 10: 'otono', 11: 'otono'}
censo_celdas['temporada4'] = censo_celdas['fecha'].dt.month.map(mapa_temporada4)
censo_celdas['celda_imputada'] = (censo_celdas['horas_validas'] == 0).astype(int)

balance_imputacion = (censo_celdas.groupby(['temporada4', 'clase'])
                      .agg(celdas=('celda_imputada', 'count'),
                           pct_imputada=('celda_imputada', lambda s: round(s.mean() * 100, 2)))
                      .reset_index())
orden_temporadas = ['invierno', 'primavera', 'verano', 'otono']
balance_imputacion['temporada4'] = pd.Categorical(balance_imputacion['temporada4'], orden_temporadas)
balance_imputacion = balance_imputacion.sort_values(['temporada4', 'clase'])

print('\n% FULLY IMPUTED CELLS BY SEASON AND REGIME:')
print(balance_imputacion.to_string(index=False))
balance_imputacion.to_csv(censo_dir / 'balance_imputacion_temporada_regimen.csv', index=False)
print(f"\nSaved to: {censo_dir / 'balance_imputacion_temporada_regimen.csv'}")


% FULLY IMPUTED CELLS BY SEASON AND REGIME:
temporada4  clase  celdas  pct_imputada
  invierno      0    1740          0.92
  invierno      1    6390          2.28
 primavera      0    1080          0.28
 primavera      1    7200          1.85
    verano      0    3540          1.07
    verano      1    4740          2.89
     otono      1    8190          1.49

Saved to: ../data_processed/censo_datos/balance_imputacion_temporada_regimen.csv


In [9]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(orden_temporadas))
width = 0.35
vac = balance_imputacion[balance_imputacion['clase'] == 0].set_index('temporada4').reindex(orden_temporadas)
cla = balance_imputacion[balance_imputacion['clase'] == 1].set_index('temporada4').reindex(orden_temporadas)

bars1 = ax.bar(x - width/2, vac['pct_imputada'], width,
               label='Vacation', color='#3498db', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, cla['pct_imputada'], width,
               label='School term', color='#e74c3c', alpha=0.8, edgecolor='black')

for bars, datos in [(bars1, vac), (bars2, cla)]:
    for bar, (_, fila) in zip(bars, datos.iterrows()):
        if np.isnan(bar.get_height()):
            continue
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f"{bar.get_height():.1f}%\n(n={int(fila['celdas']):,})",
                ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([{'invierno': 'Winter', 'primavera': 'Spring', 'verano': 'Summer', 'otono': 'Autumn'}[t] for t in orden_temporadas])
ax.set_xlabel('Season', fontsize=12, fontweight='bold')
ax.set_ylabel('% fully imputed cells', fontsize=12, fontweight='bold')
ax.set_title('Arrival-window imputation balance: season x regime\n(large differences between paired bars could affect the corresponding contrast)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(censo_dir / '03_balance_imputacion_temporada_regimen.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {censo_dir / '03_balance_imputacion_temporada_regimen.png'}")

Plot saved to: ../data_processed/censo_datos/03_balance_imputacion_temporada_regimen.png


## 7. Analysis days by season, calendar regime and monitoring zone

In [10]:
panel['temporada4'] = panel['mes'].map(mapa_temporada4)

conteo_temporada_regimen = (panel.groupby(['temporada4', 'clase'])['date']
                            .nunique().rename('n_dias').reset_index())
conteo_temporada_regimen['temporada4'] = pd.Categorical(conteo_temporada_regimen['temporada4'], orden_temporadas)
conteo_temporada_regimen = conteo_temporada_regimen.sort_values(['temporada4', 'clase'])

print("=" * 70)
print('ANALYSIS DAYS BY SEASON AND REGIME (final arrival-window panel)')
print("=" * 70)
print(conteo_temporada_regimen.to_string(index=False))

conteo_mes_anio = (panel.groupby(['anio', 'mes', 'clase'])['date']
                   .nunique().rename('n_dias').reset_index())
conteo_estacion = panel.groupby('estacion')['date'].nunique().rename('n_dias')

print('\nDays per monitoring zone in the final panel:')
print(conteo_estacion)

conteo_temporada_regimen.to_csv(censo_dir / 'n_dias_por_temporada_regimen.csv', index=False)

# Four categories: weekday/weekend x school term/vacation
mapa_tipo_dia = calendario.set_index('date')['tipo_dia']
panel['tipo_dia'] = panel['date'].map(mapa_tipo_dia)
conteo_tipo_dia = (panel.groupby('tipo_dia')['date'].nunique()
                   .reindex(['habil_lectivo', 'habil_vacaciones', 'finde_lectivo', 'finde_vacaciones'])
                   .rename('n_dias'))
print('\nDays by day type (weekday/weekend x school term/vacation):')
print(conteo_tipo_dia.to_string())
conteo_tipo_dia.to_csv(censo_dir / 'n_dias_por_tipo_dia.csv')

# Season x regime x weekday/weekend counts used in the manuscript sample table
panel['habil'] = np.where(panel['tipo_dia'].str.startswith('habil'), 'habil', 'finde')
tabla_temporada_tipo = (panel.groupby(['temporada4', 'clase', 'habil'])['date'].nunique()
                        .rename('n_dias').reset_index())
tabla_temporada_tipo['temporada4'] = pd.Categorical(tabla_temporada_tipo['temporada4'], orden_temporadas)
tabla_temporada_tipo = tabla_temporada_tipo.sort_values(['temporada4', 'clase', 'habil'])
tabla_ancha = (tabla_temporada_tipo.pivot_table(index='temporada4', columns=['clase', 'habil'],
                                                values='n_dias', fill_value=0, observed=False)
               .reindex(orden_temporadas))
tabla_ancha.columns = [f"{'clases' if c == 1 else 'vacaciones'}_{h}" for c, h in tabla_ancha.columns]
tabla_ancha = tabla_ancha.reindex(columns=['vacaciones_habil', 'vacaciones_finde',
                                           'clases_habil', 'clases_finde'], fill_value=0).astype(int)
tabla_ancha.loc['total'] = tabla_ancha.sum()
print('\nDays by season, regime and weekday/weekend:')
print(tabla_ancha.to_string())
tabla_ancha.to_csv(censo_dir / 'n_dias_por_temporada_regimen_habil.csv')
conteo_mes_anio.to_csv(censo_dir / 'n_dias_por_mes_anio_regimen.csv', index=False)
conteo_estacion.to_csv(censo_dir / 'n_dias_por_estacion.csv')
print(f"\nSaved to: {censo_dir / 'n_dias_por_temporada_regimen.csv'}")
print(f"Saved to: {censo_dir / 'n_dias_por_mes_anio_regimen.csv'}")
print(f"Saved to: {censo_dir / 'n_dias_por_estacion.csv'}")

ANALYSIS DAYS BY SEASON AND REGIME (final arrival-window panel)
temporada4  clase  n_dias
  invierno      0      58
  invierno      1     213
 primavera      0      36
 primavera      1     240
    verano      0     118
    verano      1     158
     otono      1     273

Days per monitoring zone in the final panel:
estacion
CENTRO       1096
NORESTE3     1096
NOROESTE3    1096
NORTE2       1096
SUR          1096
SUROESTE2    1096
Name: n_dias, dtype: int64

Days by day type (weekday/weekend x school term/vacation):
tipo_dia
habil_lectivo       628
habil_vacaciones    155
finde_lectivo       256
finde_vacaciones     57

Days by season, regime and weekday/weekend:
            vacaciones_habil  vacaciones_finde  clases_habil  clases_finde
temporada4                                                                
invierno                  43                15           152            61
primavera                 30                 6           167            73
verano                    82

In [11]:
fig, ax = plt.subplots(figsize=(12, 6))

vac_n = conteo_temporada_regimen[conteo_temporada_regimen['clase'] == 0].set_index('temporada4').reindex(orden_temporadas)
cla_n = conteo_temporada_regimen[conteo_temporada_regimen['clase'] == 1].set_index('temporada4').reindex(orden_temporadas)

bars1 = ax.bar(x - width/2, vac_n['n_dias'], width,
               label='Vacation', color='#3498db', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, cla_n['n_dias'], width,
               label='School term', color='#e74c3c', alpha=0.8, edgecolor='black')

for bars in [bars1, bars2]:
    for bar in bars:
        if np.isnan(bar.get_height()):
            continue
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{int(bar.get_height())}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([{'invierno': 'Winter', 'primavera': 'Spring', 'verano': 'Summer', 'otono': 'Autumn'}[t] for t in orden_temporadas])
ax.set_xlabel('Season', fontsize=12, fontweight='bold')
ax.set_ylabel('Analysis days', fontsize=12, fontweight='bold')
ax.set_title('How many days support each season x regime comparison?\n(within-season comparisons require bars of both colors)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(censo_dir / '04_n_dias_temporada_regimen.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {censo_dir / '04_n_dias_temporada_regimen.png'}")

Plot saved to: ../data_processed/censo_datos/04_n_dias_temporada_regimen.png


## 8. Data-census conclusions

In [12]:
pct_imputadas_global = censo_celdas['celda_imputada'].mean() * 100
diferencia_max = (balance_imputacion.pivot(index='temporada4', columns='clase', values='pct_imputada')
                  .diff(axis=1).iloc[:, -1].abs().max())

print("=" * 70)
print('DATA CENSUS CONCLUSIONS')
print("=" * 70)
print(f"1. Fully imputed arrival-window cells: {pct_imputadas_global:.1f}% of all cells")
print(f"   (y {(censo_celdas['horas_validas'] == 1).mean()*100:.1f}% additional cells with only 1 of 2 hours observed).")
print(f"2. The largest within-season difference in imputation percentage between school term")
print(f"   and vacation is {diferencia_max:.1f} percentage points.")
print(f"3. The final panel is balanced across monitoring zones (same dates).")
print(f"4. Season x regime cells supporting within-season comparisons:")
soporte = conteo_temporada_regimen.pivot(index='temporada4', columns='clase', values='n_dias')
for temporada in orden_temporadas:
    fila = soporte.loc[temporada]
    n_vac = 0 if pd.isna(fila.get(0, 0)) else int(fila.get(0, 0))
    n_cla = 0 if pd.isna(fila.get(1, 0)) else int(fila.get(1, 0))
    print(f"   - {temporada:10s}: vacation={n_vac:3d} days, school term={n_cla:3d} days")
print(f"5. The main analysis compares TWO imputation methods")
print(f"   (hybrid and MIMA); agreement is a sensitivity check, not proof of unbiasedness.")

print('\nCOMPLETED')

DATA CENSUS CONCLUSIONS
1. Fully imputed arrival-window cells: 1.8% of all cells
   (y 0.8% additional cells with only 1 of 2 hours observed).
2. The largest within-season difference in imputation percentage between school term
   and vacation is 1.8 percentage points.
3. The final panel is balanced across monitoring zones (same dates).
4. Season x regime cells supporting within-season comparisons:
   - invierno  : vacation= 58 days, school term=213 days
   - primavera : vacation= 36 days, school term=240 days
   - verano    : vacation=118 days, school term=158 days
   - otono     : vacation=  0 days, school term=273 days
5. The main analysis compares TWO imputation methods
   (hybrid and MIMA); agreement is a sensitivity check, not proof of unbiasedness.

COMPLETED
